# Crop Yield Prediction

**Tabular Regression Project** — Predict crop yield from agricultural data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Crop Yield (Kaggle)
Target: Crop yield

## 2 · Project Overview

We predict **crop yield** based on agricultural and environmental features. This is an important application of ML in **precision agriculture** — helping farmers and policymakers optimize food production.

The dataset is downloaded from Kaggle at runtime.

## 3 · Learning Objectives

1. Download agricultural datasets from Kaggle.
2. Handle mixed feature types in agricultural data.
3. Apply regression models to yield prediction.
4. Use LazyPredict and FLAML for model selection.
5. Understand agricultural ML applications.

## 4 · Problem Statement

Predict **crop yield** given environmental and agricultural features. This helps optimize farming practices and forecast food production.

## 5 · Why This Project Matters

- **Food security** depends on accurate yield forecasting.
- Precision agriculture reduces waste and increases efficiency.
- ML can process many environmental variables simultaneously.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle crop yield datasets |
| **Target** | Yield / production measure |
| **Note** | Downloaded via kagglehub at runtime |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle agricultural datasets.
- **License**: Check Kaggle dataset page for specific license.
- **Note**: Agricultural data varies by region and time period.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download('patelris/crop-yield-prediction-dataset')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'First dataset failed: {e}')
    try:
        path = kagglehub.dataset_download('abhinand05/crop-production-in-india')
        print(f'Downloaded to: {path}')
    except Exception as e2:
        print(f'Second dataset failed: {e2}')
        path = SAVE_DIR

import glob
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
data_file = sorted(csv_files, key=os.path.getsize, reverse=True)[0]
print(f'Using: {data_file}')

df = pd.read_csv(data_file)
print(f'Original shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f'Sampled to: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nDtypes:')
print(df.dtypes)

## 13 · Exploratory Data Analysis

In [ ]:
# Identify numeric and categorical columns
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Numeric columns ({len(num_cols)}): {num_cols}')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

# Plot numeric distributions
n_plot = min(4, len(num_cols))
if n_plot > 0:
    fig, axes = plt.subplots(1, n_plot, figsize=(4*n_plot, 4))
    if n_plot == 1: axes = [axes]
    for ax, col in zip(axes, num_cols[:n_plot]):
        df[col].hist(bins=30, ax=ax, edgecolor='black')
        ax.set_title(col[:20])
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
    plt.show()

## 14 · Target Analysis

In [ ]:
# Identify target column (last numeric column or one with 'yield'/'production')
target_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['yield', 'production', 'output'])]
if target_candidates:
    TARGET = target_candidates[-1]
else:
    TARGET = num_cols[-1]  # last numeric column

print(f'Target: {TARGET}')
print(df[TARGET].describe())
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing Strategy

Drop columns with too many missing values or too high cardinality. Label-encode remaining categoricals.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Drop columns with >50% missing
thresh = len(df) * 0.5
df = df.dropna(axis=1, thresh=int(thresh))

# Drop rows where target is missing
df = df.dropna(subset=[TARGET])

# Fill remaining numeric NaN with median
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

# Encode categoricals with < 100 unique values, drop high-cardinality
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 100:
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
X_tr_lp = X_train.head(min(5000, len(X_train))); y_tr_lp = y_train.head(min(5000, len(y_train)))
X_te_lp = X_test.head(min(1000, len(X_test))); y_te_lp = y_test.head(min(1000, len(y_test)))
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_tr_lp, X_te_lp, y_tr_lp, y_te_lp)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- Agricultural yield depends on multiple interacting factors.
- Region/location features often dominate — different areas have different soil and climate.
- ML models can identify which factors most influence yield in specific regions.
- Predictions can guide resource allocation and planting decisions.

## 26 · Limitations

- Dataset quality varies — agricultural data can be noisy.
- No weather/climate time series features.
- Regional differences may limit model generalization.
- Aggregated data may hide field-level variation.

## 27 · How to Improve This Project

1. Add weather/rainfall data.
2. Include soil quality features.
3. Use satellite imagery features.
4. Time-series modeling for seasonal patterns.
5. Region-specific models.

## 28 · Production Considerations

- Need real-time weather integration.
- Satellite data for field-level monitoring.
- Farmer-friendly interface for predictions.
- Model updates each growing season.

## 29 · Common Mistakes

1. Ignoring regional variation.
2. Not handling high-cardinality crop/location names.
3. Including future information (post-harvest data in features).
4. Not validating across different seasons/years.

## 30 · Mini Challenge / Exercises

1. Group analysis: which crop has the most predictable yield?
2. Try adding polynomial features for key numeric inputs.
3. Build separate models for top 3 crops by volume.
4. Investigate geographic patterns in prediction errors.

## 31 · Final Summary / Key Takeaways

- Crop yield prediction is a valuable agricultural ML application.
- Data quality and feature availability are the main challenges.
- Gradient-boosting models handle mixed feature types well.
- Domain knowledge (agronomy) should guide feature engineering.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')